# 🔍 Data Foundation | Notebook 02
## Qualidade de Dados em PySpark — Bronze → Silver

---

**Série:** #DataFoundation  
**Autor:** Léo Touguinha | Especialista em Dados | Mercado Financeiro  
**Pré-requisito:** Notebook 01 executado (data/raw/ populado)

---

## Contexto

> **Dado sem qualidade não é ativo — é passivo.**

Em ambientes financeiros regulados (BACEN, SUSEP, CVM), dados de má qualidade geram multas, modelos enviesados e decisões estratégicas baseadas em premissas falsas.

Este notebook implementa as **4 dimensões de qualidade** que qualquer Data Lead precisa dominar e — mais importante — **quantifica o impacto financeiro de cada problema encontrado.**

| Dimensão | Verifica | Peso no DQ Score |
|----------|----------|------------------|
| Completude | Nulos e campos vazios | 30% |
| Unicidade | Duplicatas | 15% |
| Integridade | Chaves estrangeiras órfãs | 30% |
| Validade | Regras de negócio | 25% |

---

## Arquitetura Medallion

```
┌──────────┐     ┌─────────────────────────────┐     ┌────────┐
│  BRONZE  │────▶│           SILVER            │────▶│  GOLD  │
│  (RAW)   │     │  Validado + Limpo + Auditado│     │  (03)  │
└──────────┘     └─────────────────────────────┘     └────────┘
                            ↑ este notebook
```

## 0. Configuração

In [1]:
import os
# Configuração local para Windows — em produção usar variável de sistema ou cluster config
# Em ambiente Linux/cloud este bloco não é necessário
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] += ";C:\\hadoop\\bin"


In [2]:
import os
# Diretório temporário local para shuffle do Spark no Windows
# Em produção Linux/cloud usar /tmp ou configuração do cluster
os.environ["SPARK_LOCAL_DIRS"] = "C:/spark-tmp"
os.makedirs("C:/spark-tmp", exist_ok=True)


In [3]:
# !pip install pyspark findspark

import sys
sys.path.append("../src")

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date
from utils import semaforo_dq, separador

spark = SparkSession.builder \
    .appName("qualidade_dados") \
    .config("spark.local.dir", "C:/spark-tmp") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark {spark.version} | App: {spark.sparkContext.appName}")

✅ Spark 4.1.1 | App: qualidade_dados


## 1. Ingestão — Camada Bronze

> **Regra de ouro:** Na camada Bronze, nunca transforme — apenas registre.  
> O dado bruto precisa existir para auditoria e reprocessamento.

In [4]:
PATH_RAW = "../data/raw/"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{PATH_RAW}transacoes_raw.csv")
)

df_clientes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{PATH_RAW}clientes.csv")
)

total_raw = df_raw.count()

print("📋 Schema da Base RAW:")
df_raw.printSchema()
print(f"\n📊 Registros Bronze  : {total_raw:,}")
print(f"   Clientes (mestra) : {df_clientes.count():,}")
df_raw.show(5, truncate=False)

📋 Schema da Base RAW:
root
 |-- id: integer (nullable = true)
 |-- id_transacao: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- tipo_transacao: string (nullable = true)
 |-- valor_reais: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- data_transacao: date (nullable = true)


📊 Registros Bronze  : 10,300
   Clientes (mestra) : 1,000
+---+-------------+-----------------+--------------+-----------+----------------+--------------+
|id |id_transacao |id_cliente       |tipo_transacao|valor_reais|canal           |data_transacao|
+---+-------------+-----------------+--------------+-----------+----------------+--------------+
|1  |TXND7D06B9221|CLI00630         |debito        |4535.18    |pix             |2024-08-17    |
|2  |TXNF2811E3CAF|CLI_INVALIDO_9303|credito       |9332.3     |internet_banking|2023-10-28    |
|3  |TXNB107A3E694|CLI00774         |debito        |4227.09    |pix             |2024-02-17    |
|4  |TXN273D4E6F50|CLI00474         |deb

## 2. Completude — Análise de Nulos

**Pergunta de negócio:** Quais campos críticos estão incompletos e qual o impacto financeiro?

Em mercado financeiro, campos nulos em transações são risco regulatório.  
BACEN Resolução 4.658 exige rastreabilidade de canais e valores.

In [5]:
null_pct = df_raw.select([
    F.round(
        F.count(F.when(F.col(c).isNull(), c)) / total_raw * 100, 2
    ).alias(c)
    for c in df_raw.columns
])

print("📋 Percentual de Nulos por Coluna (%)")
null_pct.show()

# Impacto financeiro
com_nulo = df_raw.filter(
    F.col("valor_reais").isNull() |
    F.col("canal").isNull() |
    F.col("tipo_transacao").isNull()
)
n_nulos = com_nulo.count()

print("=" * 55)
print("  DIAGNÓSTICO — Completude")
print("=" * 55)
print(f"  Registros com nulo em campo crítico : {n_nulos:,}")
print(f"  % da base afetada                  : {n_nulos/total_raw*100:.1f}%")
print()
print("  ⚠️  RISCO: Transações sem canal comprometem")
print("       relatórios de adoção digital para o BACEN.")
print("  ✅ AÇÃO: Imputação por canal mais frequente")
print("       do cliente (look-back 90 dias).")
print("=" * 55)

📋 Percentual de Nulos por Coluna (%)
+---+------------+----------+--------------+-----------+-----+--------------+
| id|id_transacao|id_cliente|tipo_transacao|valor_reais|canal|data_transacao|
+---+------------+----------+--------------+-----------+-----+--------------+
|0.0|         0.0|       0.0|          1.49|       1.68| 1.82|           0.0|
+---+------------+----------+--------------+-----------+-----+--------------+

  DIAGNÓSTICO — Completude
  Registros com nulo em campo crítico : 513
  % da base afetada                  : 5.0%

  ⚠️  RISCO: Transações sem canal comprometem
       relatórios de adoção digital para o BACEN.
  ✅ AÇÃO: Imputação por canal mais frequente
       do cliente (look-back 90 dias).


## 3. Unicidade — Detecção de Duplicatas

**Pergunta de negócio:** Estamos contando a mesma transação duas vezes na reconciliação contábil?

Duplicatas em transações financeiras geram dupla contagem de receita no DRE e risco jurídico de cobrança dupla.

In [6]:
w_dup = Window.partitionBy("id_transacao")

df_contagem = df_raw.withColumn("ocorrencias", F.count("id_transacao").over(w_dup))
duplicatas  = df_contagem.filter(F.col("ocorrencias") > 1)

n_dup_linhas = duplicatas.count()
ids_dup      = duplicatas.select("id_transacao").distinct().count()
valor_dup    = duplicatas.filter(F.col("valor_reais").isNotNull()) \
                         .agg(F.sum("valor_reais")).collect()[0][0] or 0

print("=" * 55)
print("  DIAGNÓSTICO — Unicidade")
print("=" * 55)
print(f"  Linhas duplicadas          : {n_dup_linhas:,}")
print(f"  IDs únicos com duplicação  : {ids_dup:,}")
print(f"  Valor em risco             : R$ {valor_dup:,.2f}")
print()
print("  ⚠️  CAUSA RAIZ: Reprocessamento de lote sem")
print("       controle de idempotência.")
print("  ✅ AÇÃO: Implementar MERGE por id_transacao")
print("       ao invés de INSERT simples.")
print("=" * 55)

duplicatas.select("id", "id_transacao", "valor_reais", "ocorrencias") \
          .orderBy("id_transacao").show(8, truncate=False)

  DIAGNÓSTICO — Unicidade
  Linhas duplicadas          : 600
  IDs únicos com duplicação  : 300
  Valor em risco             : R$ 2,853,864.22

  ⚠️  CAUSA RAIZ: Reprocessamento de lote sem
       controle de idempotência.
  ✅ AÇÃO: Implementar MERGE por id_transacao
       ao invés de INSERT simples.
+-----+-------------+-----------+-----------+
|id   |id_transacao |valor_reais|ocorrencias|
+-----+-------------+-----------+-----------+
|2195 |TXN00C1565E0A|2347.12    |2          |
|10089|TXN00C1565E0A|2347.12    |2          |
|1671 |TXN00EFBBF95F|14344.42   |2          |
|10282|TXN00EFBBF95F|14344.42   |2          |
|8616 |TXN02A2EE5CF0|7971.63    |2          |
|10054|TXN02A2EE5CF0|7971.63    |2          |
|7671 |TXN02F67AA077|3059.62    |2          |
|10280|TXN02F67AA077|3059.62    |2          |
+-----+-------------+-----------+-----------+
only showing top 8 rows


## 4. Integridade Referencial — Orphan Records

**Pergunta de negócio:** Temos transações de clientes que não existem no cadastro?

> Transação sem cliente identificável é potencial alerta de PLD/COAF.  
> Não é problema técnico — é risco de compliance.

In [7]:
orphans = df_raw.join(
    df_clientes.select("id_cliente"),
    on="id_cliente",
    how="left_anti"
)

n_orphans    = orphans.count()
valor_orphan = orphans.filter(F.col("valor_reais").isNotNull()) \
                      .agg(F.sum("valor_reais")).collect()[0][0] or 0

print("=" * 55)
print("  DIAGNÓSTICO — Integridade Referencial")
print("=" * 55)
print(f"  Transações sem cliente       : {n_orphans:,}")
print(f"  % do total                   : {n_orphans/total_raw*100:.2f}%")
print(f"  Valor sem rastreio           : R$ {valor_orphan:,.2f}")
print()
print("  ⚠️  RISCO CRÍTICO: Escalar para Compliance.")
print("       Potencial alerta COAF / PLD-FT.")
print("  ✅ AÇÃO: Quarentena imediata + investigação.")
print("=" * 55)

  DIAGNÓSTICO — Integridade Referencial
  Transações sem cliente       : 200
  % do total                   : 1.94%
  Valor sem rastreio           : R$ 915,690.25

  ⚠️  RISCO CRÍTICO: Escalar para Compliance.
       Potencial alerta COAF / PLD-FT.
  ✅ AÇÃO: Quarentena imediata + investigação.


## 5. Validade — Regras de Negócio

In [8]:
hoje = date.today().strftime("%Y-%m-%d")

n_neg  = df_raw.filter(F.col("valor_reais") < 0).count()
n_fut  = df_raw.filter(F.col("data_transacao") > F.lit(hoje)).count()
n_tipo = df_raw.filter(
    F.col("tipo_transacao").isNotNull() &
    ~F.col("tipo_transacao").isin(["debito", "credito"])
).count()

print("=" * 55)
print("  DIAGNÓSTICO — Validade")
print("=" * 55)
print(f"  Valores negativos          : {n_neg:,}")
print(f"  Datas futuras              : {n_fut:,}")
print(f"  Tipo de transação inválido : {n_tipo:,}")
print()
print("  ⚠️  valor_reais SEMPRE positivo — débito e")
print("       crédito são controlados pelo tipo_transacao.")
print("=" * 55)

  DIAGNÓSTICO — Validade
  Valores negativos          : 195
  Datas futuras              : 55
  Tipo de transação inválido : 0

  ⚠️  valor_reais SEMPRE positivo — débito e
       crédito são controlados pelo tipo_transacao.


## 6. DQ Score — KPI de Qualidade para o C-Level

Uma métrica única que combina as 4 dimensões ponderadas pela criticidade regulatória.

In [9]:
score_completude  = (1 - n_nulos      / total_raw) * 100
score_unicidade   = (1 - n_dup_linhas / total_raw) * 100
score_integridade = (1 - n_orphans    / total_raw) * 100
# Soma n_neg + n_fut para capturar ambos os problemas de validade
score_validade    = (1 - (n_neg + n_fut) / total_raw) * 100

dq_score = (
    score_completude  * 0.30 +
    score_unicidade   * 0.15 +
    score_integridade * 0.30 +
    score_validade    * 0.25
)

print("=" * 55)
print("  DATA QUALITY SCORE — Relatório Executivo")
print("=" * 55)
print(f"  Completude   (30%) : {score_completude:6.2f} pts")
print(f"  Unicidade    (15%) : {score_unicidade:6.2f} pts")
print(f"  Integridade  (30%) : {score_integridade:6.2f} pts")
print(f"  Validade     (25%) : {score_validade:6.2f} pts")
print(f"  {'─'*38}")
print(f"  DQ SCORE           : {dq_score:6.2f} / 100")
print(f"  STATUS             : {semaforo_dq(dq_score)}")
print()
print("  Benchmark mercado financeiro:")
print("  < 70  → Dado inutilizável em produção")
print("  70–85 → Utilizável com ressalvas")
print("  85–95 → Padrão para modelos analíticos")
print("  > 95  → Padrão para relatórios regulatórios")
print("=" * 55)

  DATA QUALITY SCORE — Relatório Executivo
  Completude   (30%) :  95.02 pts
  Unicidade    (15%) :  94.17 pts
  Integridade  (30%) :  98.06 pts
  Validade     (25%) :  97.57 pts
  ──────────────────────────────────────
  DQ SCORE           :  96.44 / 100
  STATUS             : 🟢 EXCELENTE

  Benchmark mercado financeiro:
  < 70  → Dado inutilizável em produção
  70–85 → Utilizável com ressalvas
  85–95 → Padrão para modelos analíticos
  > 95  → Padrão para relatórios regulatórios


## 7. Tratamento — Bronze → Silver

In [10]:
# Passo 1: Deduplicação
w_dedup  = Window.partitionBy("id_transacao").orderBy("id")
df_dedup = df_raw.withColumn("rn", F.row_number().over(w_dedup)) \
                 .filter(F.col("rn") == 1).drop("rn")

# Passo 2: Remover orphans
df_sem_orphan = df_dedup.join(df_clientes.select("id_cliente"), on="id_cliente", how="inner")

# Passo 3: Imputar canal por moda do cliente
canal_moda = (
    df_sem_orphan.filter(F.col("canal").isNotNull())
    .groupBy("id_cliente", "canal").count()
    .withColumn("rank", F.rank().over(
        Window.partitionBy("id_cliente").orderBy(F.desc("count"))
    ))
    .filter(F.col("rank") == 1)
    .select(F.col("id_cliente"), F.col("canal").alias("canal_moda"))
)
df_imputado = df_sem_orphan.join(canal_moda, on="id_cliente", how="left") \
    .withColumn("canal", F.when(F.col("canal").isNull(), F.col("canal_moda")).otherwise(F.col("canal"))) \
    .drop("canal_moda")

# Passo 4: Corrigir valores negativos
df_val = df_imputado.withColumn("valor_reais", F.abs(F.col("valor_reais")))

# Passo 5: Remover datas futuras
df_silver = df_val.filter(
    F.col("data_transacao").isNull() | (F.col("data_transacao") <= F.lit(hoje))
)

# Passo 6: Metadados de auditoria
df_silver = df_silver \
    .withColumn("dq_processado_em", F.current_timestamp()) \
    .withColumn("dq_versao", F.lit("silver_v1"))

total_silver = df_silver.count()

print("=" * 55)
print("  TRATAMENTO — Bronze → Silver")
print("=" * 55)
print(f"  Entrada (Bronze)   : {total_raw:,}")
print(f"  Saída   (Silver)   : {total_silver:,}")
print(f"  Removidos          : {total_raw - total_silver:,}")
print(f"  Aproveitamento     : {total_silver/total_raw*100:.1f}%")
print()
print("  [1] ✅ Deduplicação por id_transacao")
print("  [2] ✅ Remoção de orphan records")
print("  [3] ✅ Imputação de canal por moda")
print("  [4] ✅ Correção de valores negativos")
print("  [5] ✅ Remoção de datas futuras")
print("  [6] ✅ Metadados de auditoria")
print("=" * 55)

  TRATAMENTO — Bronze → Silver
  Entrada (Bronze)   : 10,300
  Saída   (Silver)   : 11,819
  Removidos          : -1,519
  Aproveitamento     : 114.7%

  [1] ✅ Deduplicação por id_transacao
  [2] ✅ Remoção de orphan records
  [3] ✅ Imputação de canal por moda
  [4] ✅ Correção de valores negativos
  [5] ✅ Remoção de datas futuras
  [6] ✅ Metadados de auditoria


## 8. Persistência

## 9. Data Dictionary — Camada Silver

Definições de negócio dos campos críticos. Em produção, este catálogo viveria em ferramenta dedicada (Dataplex, DataHub, Collibra). Aqui está intencionalmente no notebook para rastreabilidade.

| Campo | Tipo | Definição de negócio | Regra de validação | Owner da definição |
|---|---|---|---|---|
| `id_transacao` | string | Identificador único de transação — imutável após criação | Formato TXN + 10 hex, sem duplicatas | Financeiro |
| `id_cliente` | string | Chave de relacionamento com cadastro mestre | Deve existir em `clientes.csv` | Cadastro |
| `tipo_transacao` | string | Natureza do movimento financeiro | Enum: `debito`, `credito` — sem outros valores | Financeiro |
| `valor_reais` | double | Valor monetário da transação em BRL | Sempre positivo — sinal controlado por `tipo_transacao` | Financeiro |
| `canal` | string | Canal digital ou físico de origem da transação | Enum: pix, app_mobile, internet_banking, agencia, totem | Digital |
| `data_transacao` | string | Data em que a transação ocorreu | Formato YYYY-MM-DD, nunca futura, nunca anterior a 2023-01-01 | Financeiro |
| `_origem` | string | Arquivo fonte da ingestão Bronze | Preenchido automaticamente no pipeline | Engenharia de Dados |
| `_dt_processamento` | string | Timestamp de processamento para linhagem | Preenchido automaticamente no pipeline | Engenharia de Dados |

**Ownership:**

| Domínio | Data Owner | Data Steward | SLA de qualidade |
|---|---|---|---|
| Transações financeiras | Diretoria Financeira | Especialista de Dados | DQ Score ≥ 85 / Completude ≥ 95% |
| Cadastro de clientes | Compliance / DPO | Analista de Dados Sênior | Completude ≥ 98% / Zero orphans |


In [11]:
import os
from datetime import datetime

os.makedirs("../data/silver", exist_ok=True)

# Colunas de linhagem — rastreabilidade de origem e processamento
df_silver_com_linhagem = (
    df_silver
    .withColumn("_origem", F.lit("transacoes_raw.csv"))
    .withColumn("_dt_processamento", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M")))
)

# Caminho B: toPandas() para contornar limitação do winutils no Windows
# Em produção Linux/cloud: df_silver.write.format("delta").partitionBy("data_transacao").save("s3://...")
df_silver_com_linhagem.toPandas().to_csv("../data/silver/transacoes_silver.csv", index=False)

print("✅ Camada Silver salva com colunas de linhagem: data/silver/transacoes_silver.csv")
print(f"   _origem             : transacoes_raw.csv")
print(f"   _dt_processamento   : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print()
print("💡 Em produção:")
print("   df_silver.write.format('delta')")
print("   .mode('overwrite')")
print("   .partitionBy('data_transacao')")
print("   .save('s3://datalake/silver/transacoes/')")

spark.stop()
print("\n✅ SparkSession encerrada.")


✅ Camada Silver salva com colunas de linhagem: data/silver/transacoes_silver.csv
   _origem             : transacoes_raw.csv
   _dt_processamento   : 2026-06-30 15:34

💡 Em produção:
   df_silver.write.format('delta')
   .mode('overwrite')
   .partitionBy('data_transacao')
   .save('s3://datalake/silver/transacoes/')

✅ SparkSession encerrada.
